In [12]:
import os
import sys
import numpy as np
import pandas as pd
from PIL import Image
from numpy import gradient
from tqdm import tqdm
from glob import glob
from sklearn.neighbors import KDTree


class Config:
    pass

In [13]:
!uv add nbformat==5.7.0


Resolved 34 packages in 4ms
Audited 32 packages in 0.23ms


In [14]:
def read_tiff_as_array(tiff_path):
    """Reads a multipage TIFF as a numpy array (D, H, W)"""
    img = Image.open(tiff_path)
    slices = []

    for i in range(img.n_frames):
        img.seek(i)
        frame = np.array(img)
        slices.append(frame)

    return np.stack(slices, axis=0)


def read_cube_info(cube: np.ndarray):
    cube_shape = cube.shape
    cube_size = cube.nbytes / (1024**2)  #
    has_air = np.any(cube == 0)
    has_paper = np.any(cube == 1)
    has_non_label = np.any(cube == 2)

    return {
        "shape": cube_shape,
        "size_MB": cube_size,
        "has_air": has_air,
        "has_paper": has_paper,
        "has_non_label": has_non_label,
    }


cube320 = read_tiff_as_array(
    "../../input/00_original/train_labels/105068588.tif"
)  # /1004283650.tif
cube320.shape

(320, 320, 320)

In [15]:
read_cube_info(cube320)

{'shape': (320, 320, 320),
 'size_MB': 31.25,
 'has_air': np.True_,
 'has_paper': np.True_,
 'has_non_label': np.False_}

In [16]:
# train_labels_paths = glob("../../input/00_original/train_labels/*.tif")
# infos = []
# for path in tqdm(train_labels_paths):
#     sample_id = path.split("/")[-1].replace(".tif", "")
#     cube320 = read_tiff_as_array(path)
#     info = read_cube_info(cube320)
#     info["sample_id"] = sample_id
#     infos.append(info)

# pd.DataFrame(infos).to_csv("train_labels_infos.csv", index=False)

In [17]:
cube_info_df = pd.read_csv("train_labels_infos.csv")
cube_info_df.head()

,shape,size_MB,has_air,has_paper,has_non_label,sample_id
0,"(320, 320, 320)",31.25,True,True,True,2423079874
1,"(320, 320, 320)",31.25,True,True,True,787804611
2,"(320, 320, 320)",31.25,True,True,True,2456859500
3,"(320, 320, 320)",31.25,True,True,True,3294954456
4,"(256, 256, 256)",16.00,True,True,True,70695797


In [18]:
valid_sample_ids = cube_info_df[
    (cube_info_df["has_non_label"] == True)
    & (cube_info_df["shape"] == "(320, 320, 320)")
]["sample_id"].tolist()
valid_sample_ids[:5]

[2423079874, 787804611, 2456859500, 3294954456, 3040864797]

***符号付距離データの作成***

In [19]:
# 符号付距離データの作成をする関数群を作成


def format_cube2cube320(cube: np.ndarray):
    """全ての立方体の解像度を320x320x320に変換"""
    if cube.shape[0] == cube.shape[1] == cube.shape[2] == 320:
        return cube

    target_size = 320
    current_size = cube.shape[0]
    pad_width = max(0, target_size - current_size)
    pad_before = pad_width // 2
    pad_after = pad_width - pad_before

    if current_size < target_size:
        # Pad the cube
        padded_cube = np.pad(
            cube,
            ((pad_before, pad_after), (pad_before, pad_after), (pad_before, pad_after)),
            mode="constant",
            constant_values=0,
        )
        return padded_cube
    elif current_size > target_size:
        # Crop the cube
        start_idx = (current_size - target_size) // 2
        end_idx = start_idx + target_size
        cropped_cube = cube[start_idx:end_idx, start_idx:end_idx, start_idx:end_idx]
        return cropped_cube
    else:
        return cube


def compute_sdf(cube: np.ndarray, n_particles: int = 100000) -> np.ndarray:
    assert cube.ndim == 3, "Input cube must be 3-dimensional"
    assert cube.max() == 2, "ラベル無し(=2)が含まれている"

    """符号付距離関数(Signed Distance Function)を計算"""
    has_paper_mask = cube == 1  # 紙が存在する位置

    has_paper_indices = np.argwhere(has_paper_mask)

    has_paper_tree = KDTree(has_paper_indices)

    sdf = np.zeros(cube.shape, dtype=np.float32)

    # 立法体の中央を0として，立方体中のランダムな点をn_particles個サンプリング
    z_size, y_size, x_size = cube.shape
    center = np.array([z_size / 2, y_size / 2, x_size / 2])
    random_points = np.random.rand(n_particles, 3) * np.array([cube.shape])
    random_points -= center
    # 各ランダム点から最も近い紙の位置までの距離を計算
    distances, _ = has_paper_tree.query(random_points, k=1)
    distances = distances.flatten()

    for i, point in enumerate(random_points):
        z, y, x = point + center
        z, y, x = int(z), int(y), int(x)
        sdf[z, y, x] = distances[i]
    # 符号付距離関数の符号を設定
    sdf[cube == 1] *= -1  # 紙の内部は負の距離
    sdf[cube == 0] *= 1  # 空気の内部は正の距離

    # distancesにも符号を設定
    for i, point in enumerate(random_points):
        z, y, x = point + center
        z, y, x = int(z), int(y), int(x)
        if cube[z, y, x] == 1:
            distances[i] *= -1
        elif cube[z, y, x] == 0:
            distances[i] *= 1

    return sdf, random_points, distances


def normalize_sdf(sdf: np.ndarray, cube_size=320) -> np.ndarray:
    """符号付距離関数を正規化"""
    max_distance = cube_size // 2  # 最大距離を計算
    normalized_sdf = sdf / max_distance  # np.clip(sdf / max_distance, -1.0, 1.0)
    return normalized_sdf


# 可視化関連
def plot_cube320(cube320):
    pyo.init_notebook_mode(connected=True)
    import plotly.graph_objects as go

    sub = 6
    label_simple = cube320[::sub, ::sub, ::sub]
    z, y, x = np.where(label_simple == 1)

    fig = go.Figure(
        data=go.Scatter3d(
            x=x,
            y=y,
            z=z,
            mode="markers",
            marker=dict(size=2.5, color="red", opacity=0.85),
        )
    )

    fig.update_layout(
        title=f"Simplified Papyrus Surface: 1004283650",
        scene=dict(
            xaxis_title="X", yaxis_title="Y", zaxis_title="Z", aspectmode="data"
        ),
        width=750,
        height=700,
        template="plotly_dark",
    )

    fig.show()

In [20]:
idx = 99
sample_id = (
    glob("../../input/00_original/train_images/*.tif")[idx]
    .split("/")[-1]
    .replace(".tif", "")
)

In [ ]:
cubeXXX = read_tiff_as_array(f"../../input/00_original/train_labels/{sample_id}.tif")
cube320_labels = format_cube2cube320(cubeXXX)
_, random_points, distances = compute_sdf(cube320_labels)
normalize_points = normalize_sdf(random_points)
normalize_distance = normalize_sdf(distances)

In [ ]:
for val_id in tqdm(valid_sample_ids):
    base_dir = f"../../input/03_sdf_dataset/{val_id}/"
    os.makedirs(base_dir, exist_ok=True)
    cubeXXX = read_tiff_as_array(f"../../input/00_original/train_labels/{val_id}.tif")
    ct = read_tiff_as_array(f"../../input/00_original/train_images/{val_id}.tif")
    points_save_path = os.path.join(base_dir, "points.npy")
    sdf_save_path = os.path.join(base_dir, "sdf.npy")
    ct_save_path = os.path.join(base_dir, "ct320.npy")

    if os.path.exists(points_save_path) and os.path.exists(sdf_save_path):
        continue

    np.save(ct_save_path, ct)
    cube320_labels = format_cube2cube320(cubeXXX)
    _, random_points, distances = compute_sdf(cube320_labels)
    normalize_points = normalize_sdf(random_points)
    normalize_distance = normalize_sdf(distances)
    print(
        "normalize_points:",
        normalize_points.shape,
        normalize_points.max(),
        normalize_points.min(),
    )
    print(
        "normalize_distance:",
        normalize_distance.shape,
        normalize_distance.max(),
        normalize_distance.min(),
    )
    np.save(points_save_path, normalize_points)
    np.save(sdf_save_path, normalize_distance)

  0%|          | 1/742 [00:02<33:45,  2.73s/it]

normalize_points: (100000, 3) 0.9999992953392074 -0.9999952404164196
normalize_distance: (100000,) 1.8041543560147932 -1.7145827815575554


  0%|          | 2/742 [00:05<35:05,  2.84s/it]

normalize_points: (100000, 3) 0.9999841581424839 -0.9999875344290722
normalize_distance: (100000,) 1.7262540911841284 -1.714245891338442


  0%|          | 3/742 [00:08<33:51,  2.75s/it]

...

normalize_points: (100000, 3) 0.9999828747090124 -0.9999995975497725
normalize_distance: (100000,) 1.7978512067275656 -1.7105401060928418


100%|█████████▉| 741/742 [58:25<00:04,  4.73s/it]

normalize_points: (100000, 3) 0.9999988057844138 -0.9999810505183664
normalize_distance: (100000,) 1.744473245988885 -1.66180834042917


100%|██████████| 742/742 [58:31<00:00,  4.73s/it]

normalize_points: (100000, 3) 0.9999981266514698 -0.9999991449584836
normalize_distance: (100000,) 1.8050485855657783 -1.6774494606560624
